# 02 MLP Improved

This notebook trains an improved MLP based on the baseline observations.

Baseline observation:
- The simple MLP produced a usable first accuracy.
- Macro-F1 was much lower than accuracy.
- Several rare classes were predicted poorly or not at all.

Changes in this notebook:
1. More training epochs.
2. Early stopping based on validation loss.
3. Class-weighted CrossEntropyLoss to handle class imbalance.
4. Dropout in the MLP as regularization.
5. Light data augmentation for the training images.

A CNN would likely be better suited for image classification because it can exploit local spatial structure. Due to time and computational constraints, we focus on a reliable MLP-based solution and mention CNNs as future work.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\tobi\Documents\Machine-Learning\ps\project_submission


In [2]:
import torch
from torch.utils.data import DataLoader
import torch.nn as nn

from src.dataset import (
    load_labels,
    make_label_mapping,
    make_transforms,
    SignLanguageDataset,
    stratified_split,
    compute_class_weights,
)
from src.models import ImprovedMLP
from src.train_utils import (
    get_device,
    train_with_early_stopping,
    evaluate,
    plot_training_curves,
    plot_metric_curve,
    plot_confusion_matrix,
)

## Data loading and split

The split is the same as in the baseline notebook to make the comparison fair.

In [3]:
DATA_ROOT = PROJECT_ROOT / "sign_lang_train" / "sign_lang_train"

df = load_labels(DATA_ROOT)
label_to_idx, idx_to_label = make_label_mapping(df["label"])

train_df, val_df, test_df = stratified_split(df, test_size=0.2, val_size=0.2, random_state=42)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Classes:", list(label_to_idx.keys()))

Train: 6195
Validation: 1549
Test: 1936
Classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


## Improved preprocessing

For training we add light data augmentation. Validation and test data are not augmented.

In [4]:
IMAGE_SIZE = 64
BATCH_SIZE = 64

train_transform = make_transforms(image_size=IMAGE_SIZE, augment=True)
eval_transform = make_transforms(image_size=IMAGE_SIZE, augment=False)

train_dataset = SignLanguageDataset(train_df, label_to_idx=label_to_idx, transform=train_transform)
val_dataset = SignLanguageDataset(val_df, label_to_idx=label_to_idx, transform=eval_transform)
test_dataset = SignLanguageDataset(test_df, label_to_idx=label_to_idx, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Class weights

The baseline had much lower macro-F1 than accuracy. This suggests that rare classes were not learned as well as frequent classes. We therefore use inverse-frequency class weights in the loss function.

In [5]:
device = get_device()
print("Device:", device)

class_weights = compute_class_weights(train_df, label_to_idx).to(device)
print("Class weights shape:", class_weights.shape)

Device: cpu
Class weights shape: torch.Size([36])


## Improved model and training

Compared to the baseline, the improved model uses dropout and an additional hidden layer. Training uses more epochs, but early stopping keeps the best validation-loss model and stops when the validation loss no longer improves.

In [6]:
model = ImprovedMLP(
    input_size=IMAGE_SIZE * IMAGE_SIZE,
    hidden_size=512,
    num_classes=len(label_to_idx),
    dropout=0.3,
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

NUM_EPOCHS = 40
PATIENCE = 6

best_model_path = PROJECT_ROOT / "saved_models" / "mlp_improved.pt"

history = train_with_early_stopping(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    save_path=best_model_path,
)

Epoch 001/40 | train_loss=3.4811 | val_loss=2.8741 | val_acc=0.1465 | val_macro_f1=0.0743
  saved new best model to c:\Users\tobi\Documents\Machine-Learning\ps\project_submission\saved_models\mlp_improved.pt
Epoch 002/40 | train_loss=2.9284 | val_loss=2.5189 | val_acc=0.1872 | val_macro_f1=0.1252
  saved new best model to c:\Users\tobi\Documents\Machine-Learning\ps\project_submission\saved_models\mlp_improved.pt
Epoch 003/40 | train_loss=2.7909 | val_loss=2.4851 | val_acc=0.2201 | val_macro_f1=0.1550
  saved new best model to c:\Users\tobi\Documents\Machine-Learning\ps\project_submission\saved_models\mlp_improved.pt
Epoch 004/40 | train_loss=2.6645 | val_loss=2.3172 | val_acc=0.2802 | val_macro_f1=0.1902
  saved new best model to c:\Users\tobi\Documents\Machine-Learning\ps\project_submission\saved_models\mlp_improved.pt
Epoch 005/40 | train_loss=2.6087 | val_loss=2.1921 | val_acc=0.2983 | val_macro_f1=0.2246
  saved new best model to c:\Users\tobi\Documents\Machine-Learning\ps\project_

In [7]:
plot_training_curves(
    history,
    PROJECT_ROOT / "plots" / "improved_training_curve.png",
    title="Improved MLP training curve",
)

plot_metric_curve(
    history,
    PROJECT_ROOT / "plots" / "improved_val_macro_f1.png",
    metric="val_macro_f1",
)

## Test evaluation of the improved model

The best model according to validation loss is loaded from disk. This matches the project requirement that the model can be loaded from a file and evaluated without retraining.

In [8]:
best_model = ImprovedMLP(
    input_size=IMAGE_SIZE * IMAGE_SIZE,
    hidden_size=512,
    num_classes=len(label_to_idx),
    dropout=0.3,
).to(device)

best_model.load_state_dict(torch.load(best_model_path, map_location=device))

test_result = evaluate(best_model, test_loader, criterion, device, idx_to_label=idx_to_label)

print("Test accuracy:", test_result["accuracy"])
print("Test macro-F1:", test_result["macro_f1"])
print("Test weighted-F1:", test_result["weighted_f1"])
print(test_result["classification_report"])

plot_confusion_matrix(
    test_result["y_true"],
    test_result["y_pred"],
    class_names=[idx_to_label[i] for i in range(len(idx_to_label))],
    output_path=PROJECT_ROOT / "plots" / "improved_confusion_matrix.png",
    title="Improved MLP confusion matrix",
)

Test accuracy: 0.48088842975206614
Test macro-F1: 0.43566049323871564
Test weighted-F1: 0.49359334650425274
              precision    recall  f1-score   support

           0       0.46      0.38      0.42       112
           1       0.17      0.77      0.28        22
           2       0.08      0.05      0.06        22
           3       0.33      0.45      0.38        22
           4       0.83      0.39      0.53       112
           5       0.54      0.61      0.57        23
           6       0.73      0.29      0.41       112
           7       0.25      0.59      0.36        22
           8       0.30      0.50      0.37        34
           9       0.73      0.71      0.72       112
           a       0.11      0.05      0.06        22
           b       0.38      0.68      0.48        56
           c       0.79      0.52      0.63       112
           d       0.33      0.38      0.35        34
           e       0.46      0.48      0.47        23
           f       0.27    

## Short documentation of changes

Use the results to fill in this summary:

- Baseline result:
  - Accuracy:
  - Macro-F1:
- Improved result:
  - Accuracy:
  - Macro-F1:

Interpretation:
- If macro-F1 improves, class weights and regularization helped rare classes.
- If accuracy improves but macro-F1 remains low, class imbalance or model limitations remain.
- If validation loss stopped improving, early stopping prevented unnecessary overtraining.
- A CNN remains a plausible future improvement because it would exploit spatial image structure instead of flattening pixels.